# Fake News Prediction

## 0 : Fake News
## 1 : Real News

In [112]:
# import files

import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.metrics import accuracy_score , f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
import string
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier , RandomForestClassifier

# Data preprocessing

In [71]:
# loading data sets in pandas data set
df_fake = pd.read_csv(r"C:\Users\91881\Documents\ML_Projects\Fake (1).csv")
df_true = pd.read_csv(r"C:\Users\91881\Documents\ML_Projects\True.csv")

In [72]:
df_fake.head(2)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"


In [73]:
df_true.head(2)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"


In [74]:
df_fake.shape

(23481, 4)

In [75]:
df_true.shape

(21417, 4)

In [76]:
df_fake['class'] = 0
df_true['class'] = 1

In [77]:
df_fake_manual_testing = df_fake.tail(10)
for i in range(23480,23470, -1):
    df_fake.drop([i], axis = 0, inplace = True)
df_true_manual_testing = df_fake.tail(10)
for i in range(21416,21406, -1):
    df_true.drop([i], axis = 0, inplace = True)

In [78]:
df_fake.shape , df_true.shape

((23471, 5), (21407, 5))

In [79]:
df_fake_manual_testing['class'] = 0
df_true_manual_testing['class'] = 1

C:\Users\91881\AppData\Local\Temp\ipykernel_16408\1523065411.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fake_manual_testing['class'] = 0
C:\Users\91881\AppData\Local\Temp\ipykernel_16408\1523065411.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_true_manual_testing['class'] = 1


In [80]:
df_fake_manual_testing.head(10)

,title,text,subject,date,class
23471,Seven Iranians freed in the prisoner swap have...,"21st Century Wire says This week, the historic...",Middle-east,"January 20, 2016",0
23472,#Hashtag Hell & The Fake Left,By Dady Chery and Gilbert MercierAll writers ...,Middle-east,"January 19, 2016",0
23473,Astroturfing: Journalist Reveals Brainwashing ...,Vic Bishop Waking TimesOur reality is carefull...,Middle-east,"January 19, 2016",0
23474,The New American Century: An Era of Fraud,Paul Craig RobertsIn the last years of the 20t...,Middle-east,"January 19, 2016",0
23475,Hillary Clinton: ‘Israel First’ (and no peace ...,Robert Fantina CounterpunchAlthough the United...,Middle-east,"January 18, 2016",0
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",0


In [81]:
df_true_manual_testing.head(10)

,title,text,subject,date,class
23461,REPORT: ‘Federal Government Escalated the Viol...,KILLED: Rancher and protest spokesman Robert ...,Middle-east,"January 28, 2016",1
23462,"BOILER ROOM – Oregon Standoff, Cuddle Parties,...",Tune in to the Alternate Current Radio Network...,Middle-east,"January 28, 2016",1
23463,"Eyewitness Says Feds Ambushed Bundys, 100 Shot...",Patrick Henningsen 21st Century Wire UPDATE: 1...,Middle-east,"January 27, 2016",1
23464,Episode #119 – SUNDAY WIRE: ‘You Know the Dril...,Episode #119 of SUNDAY WIRE SHOW finally resum...,Middle-east,"January 24, 2016",1
23465,‘There’ll be boots on the ground’: US making n...,21st Century Wire says Various parties in Wash...,Middle-east,"January 23, 2016",1
23466,Boston Brakes? How to Hack a New Car With Your...,21st Century Wire says For those who still ref...,Middle-east,"January 22, 2016",1
23467,Oregon Governor Says Feds ‘Must Act’ Against P...,"21st Century Wire says So far, after nearly 20...",Middle-east,"January 21, 2016",1
23468,Ron Paul on Burns Oregon Standoff and Jury Nul...,21st Century Wire says If you ve been followin...,Middle-east,"January 21, 2016",1
23469,BOILER ROOM: As the Frogs Slowly Boil – EP #40,Tune in to the Alternate Current Radio Network...,Middle-east,"January 20, 2016",1
23470,Arizona Rancher Protesting in Oregon is Target...,RTOne of the most visible members of the armed...,Middle-east,"January 20, 2016",1


In [82]:
df_merge = pd.concat([df_fake , df_true] , axis = 0)

In [83]:
df_merge.columns

Index(['title', 'text', 'subject', 'date', 'class'], dtype='object')

In [84]:
df = df_merge.drop(['title' , 'subject' , 'date'], axis = 1)

In [85]:
df.isna().sum()

text     0
class    0
dtype: int64

In [86]:
df = df.sample( frac = 1)

In [87]:
df.head(12)

,text,class
8835,WASHINGTON (Reuters) - Donald Trump endured so...,1
11651,President Donald Trump is a man in a hurry. Th...,0
20060,Donald Trump has been crucified by the leftist...,0
4263,JAKARTA (Reuters) - Washington has billed Vice...,1
18328,"Meanwhile, in virtually every media outlet acr...",0
7426,The Stupid Part of America is riled up again a...,0
10583,WASHINGTON (Reuters) - The Secret Service is i...,1
19394,MEXICO CITY (Reuters) - Tears flowed at Our La...,1
14595,MEXICO CITY (Reuters) - Mexico on Thursday und...,1
2514,MOSCOW (Reuters) - The U.S. House of Represent...,1


In [88]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.&*?\]' , '', text)
    text = re.sub("\\W" , " ", text)
    text = re.sub('https?://\S+|/S+', '', text)
    text = re.sub('<.*?>'  , '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [89]:
df['text'] = df['text'].apply(wordopt)

In [90]:
x = df['text']
y = df['class']

In [91]:
# train test split

x_train, x_test, y_train , y_test = train_test_split(x , y , test_size= .4)

In [92]:
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)


In [93]:
# Loigistic Regression
LR = LogisticRegression()
LR.fit(xv_train , y_train)

LogisticRegression()

In [94]:
pre_lr = LR.predict(xv_test)

In [95]:
LR.score(xv_test, y_test)

0.9849598930481284

In [96]:
print(classification_report(y_test , pre_lr))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      9369
           1       0.98      0.99      0.98      8583

    accuracy                           0.98     17952
   macro avg       0.98      0.99      0.98     17952
weighted avg       0.98      0.98      0.98     17952



In [98]:
# Decision tree
DT = DecisionTreeClassifier()
DT.fit(xv_train , y_train)


DecisionTreeClassifier()

In [100]:
pre_dt = DT.predict(xv_test)

In [103]:
DT.score(xv_test , y_test)

0.995153743315508

In [101]:
print(classification_report(y_test , pre_dt))

              precision    recall  f1-score   support

           0       0.99      1.00      1.00      9369
           1       1.00      0.99      0.99      8583

    accuracy                           1.00     17952
   macro avg       1.00      1.00      1.00     17952
weighted avg       1.00      1.00      1.00     17952



In [107]:
GB = GradientBoostingClassifier(random_state= 0)
GB.fit(xv_train, y_train)

GradientBoostingClassifier(random_state=0)

In [109]:
pre_gb = GB.predict(xv_test)

In [110]:
GB.score(xv_test, y_test)

0.9948195187165776

In [111]:
print(classification_report(y_test , pre_gb))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      9369
           1       0.99      1.00      0.99      8583

    accuracy                           0.99     17952
   macro avg       0.99      0.99      0.99     17952
weighted avg       0.99      0.99      0.99     17952



In [113]:
RFC = RandomForestClassifier( random_state= 0)

In [115]:
RFC.fit(xv_train , y_train)
pre_rfc = RFC.predict(xv_test)

In [116]:
RFC.score(xv_test , y_test)

0.9874108734402852

In [117]:
print(classification_report(y_test , pre_rfc))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      9369
           1       0.99      0.99      0.99      8583

    accuracy                           0.99     17952
   macro avg       0.99      0.99      0.99     17952
weighted avg       0.99      0.99      0.99     17952



In [118]:
def output_label(n):
    if n == 0:
        return  'FAKE NEWS!!!'
    else:
        return  'REAL NEWS!!!'

In [169]:
def manual_testing(news):
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test['text'] = new_def_test['text'].apply(wordopt)

    
    new_xv_test = vectorization.transform(new_def_test['text'])
    pre_lr = LR.predict(new_xv_test)[0]
    pre_dt = DT.predict(new_xv_test)[0]
    pre_gb = GB.predict(new_xv_test)[0]
    pre_rfc = RFC.predict(new_xv_test)[0]
    return print(
        f"\n\n LR Prediction: {output_label(pre_lr)}"
        f"\n DT Prediction: {output_label(pre_dt)}"
        f"\n GB Prediction: {output_label(pre_gb)}"
        f"\n RFC Prediction: {output_label(pre_rfc)}"
    )



In [177]:
news = input()

manual_testing(news)

 WASHINGTON (Reuters) - The U.S. Congress on Thursday averted a government shutdown just one day before federal funding was due to expire, sending President Donald Trump a bill to provide just enough money to keep agencies operating through Jan. 19. With lawmakers eager to begin a holiday recess until Jan. 3, the House of Representatives and Senate scurried to pass the hastily written bill by votes of 231-188 and 66-32, respectively. When Congress returns, lawmakers will immediately have to get back to work on appropriating more money for a fiscal year that already will be three months old. They will try to pass an â€œomnibusâ€ spending bill to fund the government from Jan. 19 through Sept. 30. Negotiators have been struggling for months over thorny issues such as the amount of defense-spending increases versus increases for other domestic programs, including medical research, opioid treatment and â€œanti-terrorismâ€ activities. Fiscal hawks, meanwhile, are angry that Congress is aga



 LR Prediction: REAL NEWS!!!
 DT Prediction: REAL NEWS!!!
 GB Prediction: REAL NEWS!!!
 RFC Prediction: REAL NEWS!!!
